In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [31]:
%pip install -q cooler

In [32]:
from pathlib import Path, PurePosixPath
import torch, zipfile, shutil, cooler

In [6]:
%cd /content/drive/MyDrive

!git clone https://github.com/OluwadareLab/UGR-HiC.git

/content/drive/MyDrive
Cloning into 'UGR-HiC'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 0), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 6.84 MiB | 9.81 MiB/s, done.


In [7]:
%cd /content/drive/MyDrive/UGR-HiC

!git clone https://github.com/OluwadareLab/HiCForecast.git HiCForecast-main

/content/drive/MyDrive/UGR-HiC
Cloning into 'HiCForecast-main'...
remote: Enumerating objects: 5494, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 5494 (delta 56), reused 52 (delta 29), pack-reused 5407 (from 1)
Receiving objects: 100% (5494/5494), 9.55 GiB | 15.37 MiB/s, done.
Resolving deltas: 100% (1590/1590), done.
Updating files: 100% (43/43), done.


In [11]:
project = Path("/content/drive/MyDrive/UGR-HiC")

print("UGR-HiC:", (project / "main.ipynb").exists())
print("HiCForecast:", (project / "HiCForecast-main/scripts/inference.py").exists())
print("Pretrained model:", (project / "HiCForecast-main/trained_model/HiCForecast.pkl").exists())

UGR-HiC: True
HiCForecast: True
Pretrained model: True


In [12]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Please enable GPU in Colab Runtime settings.")

CUDA Available: True
GPU: Tesla T4


In [14]:
# Create a download folder
DOWNLOAD_DIR = Path("/content/drive/MyDrive/UGR-HiC/downloads")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Download directory:", DOWNLOAD_DIR)

Download directory: /content/drive/MyDrive/UGR-HiC/downloads


In [15]:
# Download the dataset
!wget -c --show-progress \
    "https://zenodo.org/records/14531696/files/hicforecast_raw.zip?download=1" \
    -O "/content/drive/MyDrive/UGR-HiC/downloads/hicforecast_raw.zip"

--2026-11-06 23:42:33--  https://zenodo.org/records/14531696/files/hicforecast_raw.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.47.192, 188.185.45.63, 137.138.121.78, ...
Connecting to zenodo.org (zenodo.org)|188.185.47.192|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 815631736 (778M) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/UGR-HiC/downloads/hicforecast_raw.zip’

/content/drive/MyDr 100%[===================>] 777.85M  12.7MB/s    in 7m 22s  

2026-11-06 23:49:56 (1.76 MB/s) - ‘/content/drive/MyDrive/UGR-HiC/downloads/hicforecast_raw.zip’ saved [815631736/815631736]



In [23]:
# verify download file
zip_path = Path(
    "/content/drive/MyDrive/UGR-HiC/downloads/hicforecast_raw.zip"
)

print("File exists:", zip_path.is_file())

if zip_path.is_file():
    print("File size:", round(zip_path.stat().st_size / 1024**2, 2), "MB")

File exists: True
File size: 777.85 MB


In [19]:
with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

    print("Total entries:", len(files))
    print("\nFirst 50 entries:\n")

    for filename in files[:50]:
        print(filename)

    cool_files = [
        name for name in files
        if name.lower().endswith(".cool")
    ]

    print("\nTotal .cool files:", len(cool_files))

Total entries: 44

First 50 entries:

raw/
raw/6/
raw/6/s12.cool
raw/6/s23.cool
raw/6/s15.cool
raw/6/s10.cool
raw/6/s9.cool
raw/6/s8.cool
raw/1/
raw/1/ICM.cool
raw/1/early_2cell.cool
raw/1/mESC_500.cool
raw/1/PN5.cool
raw/1/8cell.cool
raw/1/late_2cell.cool
raw/4/
raw/4/D4.cool
raw/4/D8.cool
raw/4/D2.cool
raw/4/D6.cool
raw/4/PSC.cool
raw/4/Ba.cool
raw/4/B.cool
raw/3/
raw/3/morula.cool
raw/3/blastocyst.cool
raw/3/2-cell.cool
raw/3/hESC.cool
raw/3/6-week.cool
raw/3/8-cell.cool
raw/2/
raw/2/E3.5.cool
raw/2/E7.5.cool
raw/2/4-cell.cool
raw/2/zygote.cool
raw/2/2-cell.cool
raw/2/8-cell.cool
raw/5/
raw/5/st27.cool
raw/5/st11.cool
raw/5/st13.cool
raw/5/st12.cool
raw/5/st18.cool
raw/5/st14.cool

Total .cool files: 37


In [21]:
# extract all dataset
OUTPUT_DIR = project / "raw_original_datasets"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [30]:
extracted = 0
skipped = 0
OVERWRITE_MISMATCH = True

with zipfile.ZipFile(zip_path, "r") as z:
    for info in z.infolist():

        if info.is_dir():
            continue

        parts = PurePosixPath(info.filename).parts

        # Expected format: raw/<dataset_id>/<filename>.cool
        if not (
            len(parts) == 3
            and parts[0] == "raw"
            and parts[1] in {"1", "2", "3", "4", "5", "6"}
            and parts[2].lower().endswith(".cool")
        ):
            continue

        dataset_id = parts[1]
        filename = parts[2]

        dataset_dir = OUTPUT_DIR / dataset_id
        dataset_dir.mkdir(parents=True, exist_ok=True)

        destination = dataset_dir / filename

        if destination.exists():
            if destination.stat().st_size != info.file_size:
                if OVERWRITE_MISMATCH:
                    print(f"SIZE MISMATCH: Overwriting {dataset_id}/{filename} (Local: {destination.stat().st_size} vs Zip: {info.file_size})")
                    destination.unlink()
                else:
                    print(f"SIZE MISMATCH - SKIPPING: Dataset {dataset_id}/{filename}")
                    skipped += 1
                    continue
            else:
                print(f"SKIPPED: Dataset {dataset_id}/{filename}")
                skipped += 1
                continue

        with z.open(info) as src, destination.open("xb") as dst:
            shutil.copyfileobj(src, dst)

        print(f"EXTRACTED: Dataset {dataset_id}/{filename}")
        extracted += 1

print("\n" + "=" * 50)
print("Extraction completed!")
print(f"Newly extracted: {extracted}")
print(f"Already present: {skipped}")
print(f"Total processed: {extracted + skipped}")
print("=" * 50)

SIZE MISMATCH: Overwriting 6/s12.cool (Local: 0 vs Zip: 10912075)
EXTRACTED: Dataset 6/s12.cool
EXTRACTED: Dataset 6/s23.cool
EXTRACTED: Dataset 6/s15.cool
EXTRACTED: Dataset 6/s10.cool
EXTRACTED: Dataset 6/s9.cool
EXTRACTED: Dataset 6/s8.cool
EXTRACTED: Dataset 1/ICM.cool
EXTRACTED: Dataset 1/early_2cell.cool
EXTRACTED: Dataset 1/mESC_500.cool
EXTRACTED: Dataset 1/PN5.cool
EXTRACTED: Dataset 1/8cell.cool
EXTRACTED: Dataset 1/late_2cell.cool
EXTRACTED: Dataset 4/D4.cool
EXTRACTED: Dataset 4/D8.cool
EXTRACTED: Dataset 4/D2.cool
EXTRACTED: Dataset 4/D6.cool
EXTRACTED: Dataset 4/PSC.cool
EXTRACTED: Dataset 4/Ba.cool
EXTRACTED: Dataset 4/B.cool
EXTRACTED: Dataset 3/morula.cool
EXTRACTED: Dataset 3/blastocyst.cool
EXTRACTED: Dataset 3/2-cell.cool
EXTRACTED: Dataset 3/hESC.cool
EXTRACTED: Dataset 3/6-week.cool
EXTRACTED: Dataset 3/8-cell.cool
EXTRACTED: Dataset 2/E3.5.cool
EXTRACTED: Dataset 2/E7.5.cool
EXTRACTED: Dataset 2/4-cell.cool
EXTRACTED: Dataset 2/zygote.cool
EXTRACTED: Dataset 2/2-

In [33]:
DATA_ROOT = Path(
    "/content/drive/MyDrive/UGR-HiC/raw_original_datasets"
)

print("=" * 75)
print("UGR-HiC DATASET VERIFICATION")
print("=" * 75)

total_files = 0
passed = 0

for dataset_id in range(1, 7):
    dataset_dir = DATA_ROOT / str(dataset_id)
    files = sorted(dataset_dir.glob("*.cool"))

    print(f"\nDATASET {dataset_id} — {len(files)} files")
    print("-" * 75)

    for file in files:
        total_files += 1

        try:
            c = cooler.Cooler(str(file))

            resolution = c.binsize
            has_chr16 = "chr16" in c.chromnames
            has_chr19 = "chr19" in c.chromnames

            resolution_ok = resolution == 40000

            if resolution_ok:
                passed += 1

            print(
                f"{file.name:<22} "
                f"Resolution: {str(resolution):<8} "
                f"chr16: {str(has_chr16):<5} "
                f"chr19: {str(has_chr19):<5} "
                f"{'OK' if resolution_ok else 'CHECK'}"
            )

        except Exception as e:
            print(f"{file.name}: ERROR - {e}")

print("\n" + "=" * 75)
print(f"Total files checked: {total_files}")
print(f"Files with 40-kb resolution: {passed}")
print("=" * 75)

UGR-HiC DATASET VERIFICATION

DATASET 1 — 6 files
---------------------------------------------------------------------------
8cell.cool             Resolution: 40000    chr16: True  chr19: True  OK
ICM.cool               Resolution: 40000    chr16: True  chr19: True  OK
PN5.cool               Resolution: 40000    chr16: True  chr19: True  OK
early_2cell.cool       Resolution: 40000    chr16: True  chr19: True  OK
late_2cell.cool        Resolution: 40000    chr16: True  chr19: True  OK
mESC_500.cool          Resolution: 40000    chr16: True  chr19: True  OK

DATASET 2 — 6 files
---------------------------------------------------------------------------
2-cell.cool            Resolution: 40000    chr16: True  chr19: True  OK
4-cell.cool            Resolution: 40000    chr16: True  chr19: True  OK
8-cell.cool            Resolution: 40000    chr16: True  chr19: True  OK
E3.5.cool              Resolution: 40000    chr16: True  chr19: True  OK
E7.5.cool              Resolution: 40000    chr